# 03 — Canonical Decision-Level Football Landscape (single-match prototype)

**Scientific purpose:** Generate one row per observed **player-possession start** in match `1874553`, with reproducible source keys, period/frame joins, team-role positions, detection flags and contemporaneous event references. The output is an **analysis scaffold**, not yet a trained dataset or validated spatial opportunity model.

**Non-negotiables:** original SkillCorner data read-only; all generated artifacts go to project-root `outputs/`; no external match data; never treat realized carry, phase-ending events, full-length pressing-chain indicators, later EPV, or future-derived source fields as available pre-action predictors. This notebook does **not** infer missing coordinates, missing teammates, or hypothetical carry outcomes.

**Scope:** one match only; freeze and scale to all 20 matches after targeted review. Tracking is loaded at selected frames via streaming, not into memory as a full-match DataFrame.

## 1 · Paths and configuration

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, json, math

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "data" / "matches").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook within the SkillCorner repository/notebooks folder")
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
MATCH_ID = "1874553"
MATCH = ROOT / "data" / "matches" / MATCH_ID
print("Repository:", ROOT)
print("Output destination:", OUT)
print("Match:", MATCH_ID)

Repository: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer
Output destination: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs
Match: 1874553


## 2 · Source keys and phase intervals

All 132 previously observed duplicate phase joins occur at shared starts/ends. We adopt **half-open phase intervals** `[frame_start, frame_end)` as an explicit *analytical convention*, not a verified SkillCorner definition. A frame exactly equal to the next phase's start belongs to the next phase. If an unmatched frame remains, we retain `null` rather than guess.

Source keys: `match_id + period + frame`; phase `index` is scoped to its match; possession `event_id` is scoped to its match. We additionally require agreement between possession `team_id` and phase `team_in_possession_id` before accepting a phase label.

In [2]:
def read_csv(path):
    with path.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def to_int(x):
    try: return int(x)
    except (TypeError, ValueError): return None

def to_float(x):
    try:
        v=float(x)
        return v if math.isfinite(v) else None
    except (TypeError, ValueError): return None

def cleaned(x):
    return None if x is None or str(x).strip()=="" else str(x).strip()

events = read_csv(MATCH / f"{MATCH_ID}_dynamic_events.csv")
phases = read_csv(MATCH / f"{MATCH_ID}_phases_of_play.csv")
possessions = [e for e in events if e.get("event_type")=="player_possession"]
options = [e for e in events if e.get("event_type")=="passing_option"]
engagements = [e for e in events if e.get("event_type")=="on_ball_engagement"]
print("Events",len(events),"Phases",len(phases),"Possessions",len(possessions))

def phase_for(e):
    t,p=to_int(e.get("frame_start")),to_int(e.get("period"))
    if t is None or p is None: return None, "invalid_frame"
    candidate=[r for r in phases if to_int(r.get("period"))==p
               and to_int(r.get("frame_start")) is not None
               and to_int(r.get("frame_end")) is not None
               and to_int(r["frame_start"])<=t<to_int(r["frame_end"])]
    if len(candidate)!=1: return None, "none" if not candidate else "multiple"
    r=candidate[0]
    if cleaned(e.get("team_id")) != cleaned(r.get("team_in_possession_id")):
        return None, "team_disagreement"
    return r, "ok"

phase_counts=Counter(phase_for(e)[1] for e in possessions)
print("Half-open, team-confirmed phase status:",dict(phase_counts))

Events 4915 Phases 423 Possessions 959
Half-open, team-confirmed phase status: {'ok': 950, 'team_disagreement': 9}


## 3 · Contemporaneous annotations (not automatically predictive features)

Passing options and engagements can overlap the possession start; their presence or identity still needs association and time-semantics checks before using fields from their records as pre-action inputs. We retain **IDs and descriptive counts**; this notebook does not use outcome/terminal annotations for forecasting. When available, the vendor's `associated_player_possession_event_id` is checked alongside temporal overlap to prevent conflating unrelated concurrent events. No zero count here is proof that no physical passing option existed.

In [3]:
def at_frame(r,period,frame):
    a,b=to_int(r.get("frame_start")),to_int(r.get("frame_end"))
    return (to_int(r.get("period"))==period and a is not None and b is not None
            and a<=frame<=b)  # diagnostic membership; event ends may be inclusive

by_period_option=defaultdict(list)
by_period_engagement=defaultdict(list)
for r in options: by_period_option[to_int(r.get("period"))].append(r)
for r in engagements: by_period_engagement[to_int(r.get("period"))].append(r)

rows=[]
for e in possessions:
    period,frame=to_int(e.get("period")),to_int(e.get("frame_start"))
    if period is None or frame is None: continue
    phase,status=phase_for(e)
    p_id=cleaned(e.get("event_id"))
    covered_options=[r for r in by_period_option[period] if at_frame(r,period,frame)]
    covered_eng=[r for r in by_period_engagement[period] if at_frame(r,period,frame)]
    # Vendor association ID is match-scoped; may be absent. Missing != negative.
    linked_options=[r for r in covered_options
                    if cleaned(r.get("associated_player_possession_event_id"))==p_id]
    rows.append({
        "match_id":MATCH_ID,"period":period,"frame":frame,
        "possession_event_id":p_id,
        "carrier_player_id":cleaned(e.get("player_id")),
        "attacking_team_id":cleaned(e.get("team_id")),
        "phase_status":status,
        "phase_index":to_int(phase.get("index")) if phase else None,
        "defending_phase_annotation":cleaned(phase.get("team_out_of_possession_phase_type")) if phase else None,
        "overlapping_option_event_count":len(covered_options),
        "linked_option_event_count":len(linked_options),
        "linked_option_event_ids":[cleaned(r.get("event_id")) for r in linked_options],
        "overlapping_engagement_event_count":len(covered_eng),
        "overlapping_engagement_event_ids":[cleaned(r.get("event_id")) for r in covered_eng],
        "observed_carry_outcome":cleaned(e.get("carry")), # label, NEVER predictor
        "observed_end_type":cleaned(e.get("end_type")), # label, NEVER predictor
    })
print("Decision candidates:",len(rows))
print("Covered passing option count (zero is not no feasible pass):",Counter(r["overlapping_option_event_count"] for r in rows).most_common(7))
print("Linked option count:",Counter(r["linked_option_event_count"] for r in rows).most_common(7))

Decision candidates: 959
Covered passing option count (zero is not no feasible pass): [(1, 338), (2, 311), (0, 157), (3, 111), (4, 32), (5, 9), (6, 1)]
Linked option count: [(1, 338), (2, 311), (0, 157), (3, 111), (4, 32), (5, 9), (6, 1)]


## 4 · Stream exact tracking frames once; record validity and team roles

Every player is keyed by `player_id`, and team roles come from each player's tracking `team_id` when available. If team identifiers cannot be established with confidence, role counts remain **unresolved**; we do not infer teams from the side of the pitch or alternate jersey ordering. For efficiency, this notebook retains compact contemporaneous positions for each possession, **not** full tracking trajectories.

**Spatial limitations:** a finite extrapolated coordinate is not a measured detection; missingness and detection flags remain explicit. No filtering to complete 22-player observations is silently applied.

In [4]:
wanted={(r["period"],r["frame"]) for r in rows}
frames={}
source_track=MATCH / f"{MATCH_ID}_tracking_extrapolated.jsonl"
with source_track.open(encoding="utf-8") as f:
    for line in f:
        fr=json.loads(line)
        k=(to_int(fr.get("period")),to_int(fr.get("frame")))
        if k in wanted: frames[k]=fr
        if len(frames)==len(wanted): break
print("Distinct requested period/frame keys:",len(wanted))
print("Exact matched period/frame keys:",len(frames))

def xy(rec):
    if not isinstance(rec,dict): return None
    x,y=to_float(rec.get("x")),to_float(rec.get("y"))
    return [x,y] if x is not None and y is not None else None

for r in rows:
    fr=frames.get((r["period"],r["frame"]))
    r["frame_found"]=fr is not None
    r["carrier_detected"]=None
    r["carrier_xy"]=None
    r["ball_detected"]=None
    r["ball_xy"]=None
    r["team_roles_resolved"]=False
    r["teammates_xy_detected"] = []
    r["opponents_xy_detected"] = []
    r["teammates_xy_extrapolated"] = []
    r["opponents_xy_extrapolated"] = []
    r["unresolved_players"] = 0
    if fr is None: continue
    pdata=fr.get("player_data") or []
    carrier=next((p for p in pdata if cleaned(p.get("player_id"))==r["carrier_player_id"]),None)
    r["carrier_xy"]=xy(carrier)
    r["carrier_detected"]=carrier.get("is_detected") if carrier else None
    ball=fr.get("ball_data") or {}
    r["ball_xy"]=xy(ball)
    r["ball_detected"]=ball.get("is_detected")
    # Determine whether tracking team_id is actually present, and matches event team_id.
    teams={cleaned(p.get("team_id")) for p in pdata if cleaned(p.get("team_id"))}
    r["team_roles_resolved"]=(len(teams)==2 and r["attacking_team_id"] in teams)
    for p in pdata:
        pt=xy(p)
        if pt is None or cleaned(p.get("player_id"))==r["carrier_player_id"]: continue
        team=cleaned(p.get("team_id"))
        if not r["team_roles_resolved"] or team not in teams:
            r["unresolved_players"]+=1; continue
        own = team==r["attacking_team_id"]
        destination=("teammates" if own else "opponents") + ("_xy_detected" if p.get("is_detected") is True else "_xy_extrapolated")
        r[destination].append({"player_id":cleaned(p.get("player_id")),"xy":pt})

print("Carrier coordinate coverage:",sum(r["carrier_xy"] is not None for r in rows),"/",len(rows))
print("Carrier detected:",sum(r["carrier_detected"] is True for r in rows),"/",len(rows))
print("Team roles resolved using tracking team_id:",sum(r["team_roles_resolved"] for r in rows),"/",len(rows))
print("Ball finite coordinates:",sum(r["ball_xy"] is not None for r in rows),"/",len(rows))

Distinct requested period/frame keys: 959
Exact matched period/frame keys: 959
Carrier coordinate coverage: 956 / 959
Carrier detected: 934 / 959
Team roles resolved using tracking team_id: 0 / 959
Ball finite coordinates: 956 / 959


## 5 · Review failures before locking a canonical rule

This is a **bounded decision gate**, not a reason to restart the data audit. Team roles may require metadata joining if tracking players lack `team_id`. The notebook will not call such cases successfully classified. Likewise, snapshots with an extrapolated carrier remain inspectable but are *not* silently treated as detected.

Important: at this stage, `observed_carry_outcome` / `observed_end_type` are labels, while `defending_phase_annotation`, association references and complete event-level context are **annotations** until feature timing has been independently established. No hypothesis test should be run on this output.

In [5]:
gate={
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "match_id":MATCH_ID,
    "n_possession_start_rows":len(rows),
    "distinct_period_frame_keys":len(wanted),
    "tracking_keys_found":len(frames),
    "phase_join_status":dict(Counter(r["phase_status"] for r in rows)),
    "carrier_xy_finite":sum(r["carrier_xy"] is not None for r in rows),
    "carrier_detected":sum(r["carrier_detected"] is True for r in rows),
    "ball_xy_finite":sum(r["ball_xy"] is not None for r in rows),
    "team_roles_resolved":sum(r["team_roles_resolved"] for r in rows),
    "multi_possession_same_tracking_key":len(rows)-len(wanted),
    "restrictions":[
        "Single illustrative match, not held-out inference",
        "Half-open phase membership is an explicit analytical convention",
        "Tracking detections and extrapolations retained separately",
        "Event overlap/linked IDs are diagnostic annotations, not confirmed pre-action features",
        "No outcome-derived columns included as predictors",
        "No hypothetical carrying options computed",
        "Unknown team roles not guessed",
    ],
}
report=OUT/"P1_canonical_decision_gate.json"
report.write_text(json.dumps({"gate":gate,
    "example_rows":[{k:v for k,v in r.items() if k not in (
        "teammates_xy_detected","opponents_xy_detected",
        "teammates_xy_extrapolated","opponents_xy_extrapolated")} for r in rows[:15]]},indent=2))
# Full canonical scaffold is confined to local outputs, never commit as unrestricted raw data.
scaffold=OUT/"P1_canonical_decisions_match_1874553.jsonl"
with scaffold.open("w",encoding="utf-8") as f:
    for r in rows: f.write(json.dumps(r,separators=(",",":"))+"\n")
print("Gate report:",report)
print("Local derived decision scaffold:",scaffold)
print(json.dumps(gate,indent=2))

Gate report: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P1_canonical_decision_gate.json
Local derived decision scaffold: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P1_canonical_decisions_match_1874553.jsonl
{
  "created_utc": "2026-10-24T23:09:39.926460+00:00",
  "match_id": "1874553",
  "n_possession_start_rows": 959,
  "distinct_period_frame_keys": 959,
  "tracking_keys_found": 959,
  "phase_join_status": {
    "ok": 950,
    "team_disagreement": 9
  },
  "carrier_xy_finite": 956,
  "carrier_detected": 934,
  "ball_xy_finite": 956,
  "team_roles_resolved": 0,
  "multi_possession_same_tracking_key": 0,
  "restrictions": [
    "Single illustrative match, not held-out inference",
    "Half-open phase membership is an explicit analytical convention",
    "Tracking detections and extrapolations retained separately",
    "Event overlap/linked IDs are diagnostic annotations, not confirmed pre-action features",
    "No outcome-derived c

## 6 · Interpretation and next freeze decision

**Upload only** `outputs/P1_canonical_decision_gate.json` for the next review. The full JSONL scaffold contains individual player locations and remains local. We will resolve any team-identifier mismatch using **official match metadata**, then perform a single targeted spatial plot and formally separate pre-action covariates from vendor retrospective annotations.

**Stop rule:** Do not run another whole-dataset audit. When team/phase/frame keys and quality flags are verified on this match, move to shared reusable functions and then all 20 matches, with match-based grouping for every evaluation.